<a href="https://colab.research.google.com/github/zwelinhtet129/AAI2025/blob/2026fall/Code_Generation_with_ReACT_Prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Exercise 2: Code Generation with ReACT Prompting
# Objective: Use a ReACT-style approach (reasoning + acting) to generate
# Python code and then execute it.
# Scenario: Generate a Python function that computes the average of a list
# of numbers, with proper handling of edge cases.

import io
import contextlib
import traceback


def react_agent(prompt):
    """
    Simulates a ReACT agent responding to a prompt.
    In a real application, the Thought and the generated code would come
    from an API call to a large language model. Here the agent's cycles
    are scripted, but every code block it produces is REALLY executed below.
    """
    print(f"--- Sending Prompt to AI Agent ---\n{prompt}\n----------------------------------")


def run_code(code, tests):
    """Really executes the agent's generated code and runs the tests on it."""
    namespace = {}
    exec(code, namespace)
    average = namespace["average"]
    results = []
    for label, data in tests:
        try:
            results.append((label, average(data)))
        except Exception as e:
            results.append((label, f"ERROR: {type(e).__name__}: {e}"))
    return results


# --- Task ---
# The initial prompt is too simple. Refine it into a ReACT prompt that forces
# the agent to reason first, then generate code, run it, observe the result,
# and fix any problems before finishing.

# Initial, less effective prompt
initial_prompt = "Write a python function to compute the average of numbers."

# Your improved prompt (ReACT style)
# Tip: Spell out the cycle - Thought -> Action (code) -> Run -> Observation -> Fix.
improved_prompt = """
You are a careful Python developer who uses the ReACT method.
For the task below, follow this exact cycle and show each stage:

1. Thought: reason about the task, the inputs, the outputs, and edge cases.
2. Action: generate the Python code for the function.
3. Run: execute the code on normal input AND edge cases (empty list, bad values).
4. Observation: describe what happened when the code ran.
5. If anything failed, go back to stage 1, explain the fix, and repeat
   until all tests pass.

Task: Write a Python function average(numbers) that returns the mean of a
list of numbers. Requirements:
- Input: a list. Output: a float, or None when no valid numbers exist.
- Must not crash on an empty list.
- Must ignore non-numeric values instead of crashing.
- Include a docstring.
"""

tests = [
    ("normal list [10, 20, 30]", [10, 20, 30]),
    ("empty list []", []),
    ("list with bad values", [10, "oops", 20, None]),
]

# --- Execution ---
print("--- Initial Prompt Response ---")
react_agent(initial_prompt)

print("\n\n--- Improved (ReACT) Prompt Response ---")
react_agent(improved_prompt)

# The agent follows the ReACT prompt: Thought -> Action -> Run -> Observation

print("\n>>> Cycle 1")
print("Thought: I need average(numbers) returning the mean. "
      "Simplest version: sum divided by length. Let me write it and test it.")
code_v1 = '''
def average(numbers):
    """Return the mean of a list of numbers."""
    return sum(numbers) / len(numbers)
'''
print("Action: generated code:\n" + code_v1)
print("Run: executing code and tests...")
for label, outcome in run_code(code_v1, tests):
    print(f"  {label} -> {outcome}")
print("Observation: works on a normal list, but CRASHES on an empty list "
      "(ZeroDivisionError) and on non-numeric values (TypeError). "
      "The code is not robust enough.")

print("\n>>> Cycle 2")
print("Thought: The failures are edge cases. Fix: filter out non-numeric "
      "values first, and return None when nothing valid remains.")
code_v2 = '''
def average(numbers):
    """Return the mean of the numeric values in a list.

    Non-numeric values are ignored. Returns None if there are
    no valid numbers (e.g. an empty list).
    """
    clean = [n for n in numbers if isinstance(n, (int, float))]
    if not clean:
        return None
    return sum(clean) / len(clean)
'''
print("Action: generated fixed code:\n" + code_v2)
print("Run: executing code and tests...")
for label, outcome in run_code(code_v2, tests):
    print(f"  {label} -> {outcome}")
print("Observation: all tests pass, including the empty list and bad values. "
      "No errors. The task is complete.")


--- Initial Prompt Response ---
--- Sending Prompt to AI Agent ---
Write a python function to compute the average of numbers.
----------------------------------


--- Improved (ReACT) Prompt Response ---
--- Sending Prompt to AI Agent ---

You are a careful Python developer who uses the ReACT method.
For the task below, follow this exact cycle and show each stage:

1. Thought: reason about the task, the inputs, the outputs, and edge cases.
2. Action: generate the Python code for the function.
3. Run: execute the code on normal input AND edge cases (empty list, bad values).
4. Observation: describe what happened when the code ran.
5. If anything failed, go back to stage 1, explain the fix, and repeat
   until all tests pass.

Task: Write a Python function average(numbers) that returns the mean of a
list of numbers. Requirements:
- Input: a list. Output: a float, or None when no valid numbers exist.
- Must not crash on an empty list.
- Must ignore non-numeric values instead of crashing.
